# Tangram on oligoDSBmt

Second step of the conversion from RDS Seurat to h5ad.

In [3]:
from matplotlib import cm
import anndata as ad
import squidpy as sq
import cellcharter as cc
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import matplotlib.patches as mpatches
import math

/home/anitag/anaconda3/envs/karospace311/lib/python3.13/site-packages/docrep/decorators.py:43: SyntaxWarning: 'n_jobs' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/home/anitag/anaconda3/envs/karospace311/lib/python3.13/site-packages/docrep/decorators.py:43: SyntaxWarning: 'show_progress_bar' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)


In [4]:
import logging
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

In [1]:
import glob
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad

d = "/home/anitag/ST_Brambilla_local/data/h5ad_parts/"
genes    = open(d + "genes.txt").read().splitlines()
barcodes = open(d + "barcodes.txt").read().splitlines()
n_genes = len(genes)

def load_layer(name):
    mats = []
    for pfile in sorted(glob.glob(f"{d}{name}_*_p.bin")):
        base = pfile[:-6]
        p = np.fromfile(base + "_p.bin", dtype=np.int32)
        i = np.fromfile(base + "_i.bin", dtype=np.int32)
        x = np.fromfile(base + "_x.bin", dtype=np.float32)
        m = sp.csc_matrix((x, i, p), shape=(n_genes, len(p) - 1))
        mats.append(m.T.tocsr())          # genes x cells -> cells x genes
    return sp.vstack(mats, format="csr")

counts = load_layer("counts")
data   = load_layer("data")
assert counts.shape == data.shape == (len(barcodes), n_genes)

obs = pd.read_csv(d + "metadata.csv", index_col=0)
assert list(obs.index) == barcodes        # same cell order as the matrices

adata = ad.AnnData(X=data, obs=obs, var=pd.DataFrame(index=genes))
adata.layers["counts"] = counts
adata.obsm["X_pca"]  = pd.read_csv(d + "pca.csv",  index_col=0).loc[adata.obs_names].values
adata.obsm["X_umap"] = pd.read_csv(d + "umap.csv", index_col=0).loc[adata.obs_names].values

adata.write_h5ad("/home/anitag/ST_Brambilla_local/data/Mouse_Myeloid_atlas_v1.h5ad", compression="gzip")
print(adata)

AnnData object with n_obs × n_vars = 172497 × 56884
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'condition', 'Species', 'Region', 'percent.mt', 'Subclass_label', 'Condition', 'Tissue', 'Context_stagewise', 'Context_stagewise_Controls_merged', 'Subclass_id', 'Subclass_id_label', 'Subclass_color', 'Tissue_label', 'Tissue_id', 'Tissue_id_label', 'Tissue_color', 'Condition_label', 'Condition_color', 'Condition_id', 'Condition_id_label', 'Module_label', 'Module_id', 'Module_color', 'Module_id_label', 'relabelled_clusters', 'Context_region'
    obsm: 'X_pca', 'X_umap'
    layers: None (.X), 'counts'


In [5]:
obj = ad.read_h5ad("/run/media/anitag/E2FB-8A68/ST_Brambilla/data/adata_cellcharter.h5ad")

In [6]:
print(obj)
print("\nObservation metadata:")
print(obj.obs.shape)

print("\nGene metadata:")
print(obj.var.shape)

print("\nAvailable dimensionality reductions and other embeddings:")
print(list(obj.obsm.keys()))

print("\nAvailable layers:")
print(list(obj.layers.keys()))

print("\nAvailable pairwise matrices:")
print(list(obj.obsp.keys()))

print("\nUnstructured metadata:")
print(list(obj.uns.keys()))

AnnData object with n_obs × n_vars = 980474 × 250
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'run', 'n_genes_by_counts', 'n_counts', 'n_genes', 'leiden_0.5', 'leiden_1', 'leiden_2', 'cell_class', 'sample_id', 'age', 'sex', 'genotype', 'condition', 'cell_type', 'cell_id', 'rbd_domain_0.1', 'rbd_domain_0.2', 'rbd_domain_0.5', 'rbd_domain_1.0', 'RBD_compartment', 'RBD_compartment_simplified', 'leiden_1.5', 'leiden_2.1', 'leiden_2.3', 'leiden_2.5', 'leiden_3', 'cytetype_annotation_leiden_3', 'cytetype_cellOntologyTerm_leiden_3', 'novae_sid', 'neighborhood_valid', 'novae_leaves', 'novae_domains_15', 'novae_domains_30', 'novae_domains_25', 'novae_domains_20', 'novae_domains_50', '_scvi_batch', '_scvi_labels', 'leiden_mana_0.5', 'leiden_mana_0.1', 'cell_class_

In [9]:
obj.obs['cell_class_updated'].cat.categories.tolist()

['Astrocytes',
 'Endocrine Cells',
 'Endothelial Cells',
 'Excitatory Neurons',
 'GABAergic Neurons',
 'Medium Spiny Neurons',
 'Microglia',
 'Myocytes',
 'Neural Stem Cells',
 'Neurons',
 'Oligodendrocyte Precursor Cells',
 'Oligodendrocytes']

In [10]:
# looking for the full gene panel 
full = obj.raw.to_adata()
full.obs = obj.obs.copy()
full.obsm = obj.obsm.copy()
full.layers["counts"] = full.X.copy()   # check these are raw counts, see below

AttributeError: 'NoneType' object has no attribute 'to_adata'

In [11]:
print(obj.shape)
print(list(obj.layers))                                  # e.g. 'counts'
for k, v in obj.layers.items():
    print(k, v.shape)
print(list(obj.uns), list(obj.obsm), list(obj.varm))
print(obj.var.columns.tolist())
print(obj.obs_names[:3].tolist(), obj.var_names[:5].tolist())

(980474, 250)
['counts', 'normalized', None]
counts (980474, 250)
normalized (980474, 250)
None (980474, 250)
['X_mana_gauss_params', '_scvi_manager_uuid', '_scvi_uuid', 'cc_neighbors', 'compartment_cell_type_enrichment', 'compartment_colors', 'cytetype_annotation_leiden_3_colors', 'cytetype_jobDetails', 'cytetype_results', 'dendrogram_compartment', 'hvg', 'leiden_mana_0.1', 'leiden_mana_0.1_colors', 'leiden_mana_0.5', 'leiden_mana_0.5_colors', 'log1p', 'mana', 'novae_attrs', 'rank_genes_groups', 'spatial_neighbors'] ['X_cellcharter', 'X_mana_gauss', 'X_pca', 'X_scVI', 'X_umap', 'novae_latent', 'spatial'] ['PCs']
['n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'gene_symbols', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'in_vocabulary', 'novae_use_gene', 'highly_variable_rank', 'variances', 'variances_norm']
['aaaaakjg-1', 'aaaaikjn-1', 'aaaapoln-1'] ['9630013A20Rik', 'A2m', 'Abca4', 'Abcb1a', 'Ace']


## Tangram